In [1]:
# !mkdir whls

# !pip download keras-nightly --dest whls
# !pip download tifffile imagecodecs --dest whls

# # medic-ai (GitHub)
# !git clone https://github.com/innat/medic-ai.git
# !pip install build -q
# !cd medic-ai && python -m build

# # copy wheel to whls folder
# !cp medic-ai/dist/*.whl whls/
# !rm -r /kaggle/working/medic-ai

In [2]:
# !pip install --no-index --find-links=/kaggle/input/xyz-installer/whls \
#     keras-nightly \
#     tifffile \
#     imagecodecs \
#     medicai


In [3]:
# import medicai
# import keras
# import tifffile
# import imagecodecs
# import wrapt

# print("medicai OK")
# print("keras:", keras.__version__)
# print("wrapt:", wrapt.__version__)


In [4]:
!pip install --no-index --find-links="/kaggle/input/surface-package-scraper" -q pytorch_lightning monai albumentations imagecodecs --no-deps # "numpy==1.26.4" "scipy==1.15.3"
!pip uninstall -q -y tensorflow 

import os
import warnings
from pathlib import Path
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from tqdm.auto import tqdm
import tifffile
import zipfile

warnings.filterwarnings("ignore")

# ===========================
# SIMPLE CONFIG
# ===========================
class Config:
    # Paths
    DATA_DIR = Path("/kaggle/input/vesuvius-challenge-surface-detection")
    TRAIN_IMAGES_DIR = DATA_DIR / "train_images"
    TRAIN_LABELS_DIR = DATA_DIR / "train_labels"
    TEST_IMAGES_DIR = DATA_DIR / "test_images"
    OUTPUT_DIR = Path(".")
    
    # Model - SIMPLE
    INPUT_SIZE = (160, 160, 160)
    IN_CHANNELS = 1
    OUT_CHANNELS = 2  # Background + surface
    
    # Training - SIMPLE
    BATCH_SIZE = 2
    NUM_WORKERS = 4
    MAX_EPOCHS = 100
    LR = 1e-4
    VAL_SPLIT = 0.2
    
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("="*70)
print("VESUVIUS - Simple DynUNet")
print("="*70)
print(f"Input: {Config.INPUT_SIZE}")
print(f"Batch: {Config.BATCH_SIZE}")
print("="*70)

# ===========================
# DATASET - SIMPLE
# ===========================
class SimpleDataset(Dataset):
    def __init__(self, images_dir, labels_dir, files):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.files = files
        print(f"Dataset: {len(files)} volumes")
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        fname = self.files[idx]
        
        # Load
        img = tifffile.imread(str(self.images_dir / fname)).astype(np.float32)
        
        mask = None
        if self.labels_dir:
            mask_path = self.labels_dir / fname
            if mask_path.exists():
                mask = tifffile.imread(str(mask_path)).astype(np.uint8)
                mask = (mask > 0).astype(np.uint8)  # Binary
        
        # To tensor
        img = torch.from_numpy(img).half().div_(255.0).unsqueeze(0)
        
        if mask is not None:
            mask = torch.from_numpy(mask).long().unsqueeze(0)
        else:
            mask = torch.zeros_like(img, dtype=torch.long)
        
        return img, mask, Path(fname).stem

# ===========================
# DATAMODULE - SIMPLE
# ===========================
from monai import transforms as MT

def collate(batch):
    return batch

class SimpleDataModule(pl.LightningDataModule):
    def __init__(self, train_dir, label_dir, size, val_split, batch_size, num_workers):
        super().__init__()
        self.train_dir = train_dir
        self.label_dir = label_dir
        self.size = size
        self.val_split = val_split
        self.batch_size = batch_size
        self.num_workers = num_workers
        
        # Simple augmentations
        self.train_transform = MT.Compose([
            MT.Resized(keys=["image", "label"], spatial_size=size, mode=["trilinear", "nearest"]),
            MT.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
            MT.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
            MT.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
        ])
        
        self.val_transform = MT.Compose([
            MT.Resized(keys=["image", "label"], spatial_size=size, mode=["trilinear", "nearest"])
        ])
    
    def setup(self, stage=None):
        files = sorted([f.name for f in self.train_dir.glob("*.tif")])
        
        random.seed(42)
        random.shuffle(files)
        split_idx = int(len(files) * (1 - self.val_split))
        
        train_files = files[:split_idx]
        val_files = files[split_idx:]
        
        print(f"Train: {len(train_files)}, Val: {len(val_files)}")
        
        self.train_dataset = SimpleDataset(self.train_dir, self.label_dir, train_files)
        self.val_dataset = SimpleDataset(self.train_dir, self.label_dir, val_files)
    
    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True,
                         num_workers=self.num_workers, pin_memory=True, collate_fn=collate)
    
    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False,
                         num_workers=self.num_workers, pin_memory=True, collate_fn=collate)
    
    def on_after_batch_transfer(self, batch, dataloader_idx):
        if not isinstance(batch, list):
            return batch
        
        x_list, y_list, ids = [], [], []
        device = self.trainer.strategy.root_device if self.trainer else torch.device("cuda")
        transform = self.train_transform if self.trainer.training else self.val_transform
        
        for x, y, fid in batch:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            
            data = transform({"image": x, "label": y})
            x_list.append(data["image"])
            y_list.append(data["label"])
            ids.append(fid)
        
        return torch.stack(x_list), torch.stack(y_list), ids

# ===========================
# MODEL - SIMPLE DYNUNET
# ===========================
from monai.networks.nets import DynUNet
from monai.losses import DiceCELoss

class SimpleDynUNet(pl.LightningModule):
    def __init__(self, in_channels=1, out_channels=2, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        
        # DynUNet - automatically calculates architecture
        spatial_dims = 3
        kernel_size = [[3, 3, 3]] * 5
        strides = [[1, 1, 1]] + [[2, 2, 2]] * 4
        upsample_kernel_size = strides[1:]
        
        self.model = DynUNet(
            spatial_dims=spatial_dims,
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            strides=strides,
            upsample_kernel_size=upsample_kernel_size,
            norm_name="instance",
            deep_supervision=False,
            dropout=0.1
        )
        
        self.loss_fn = DiceCELoss(softmax=True, to_onehot_y=False, include_background=True)
        self.lr = lr
    
    def forward(self, x):
        return self.model(x)
    
    def _compute_metrics(self, preds, targets):
        preds_hard = torch.argmax(torch.softmax(preds, dim=1), dim=1)
        
        # Dice
        pred_fg = (preds_hard == 1).float()
        target_fg = (targets.squeeze(1) == 1).float()
        
        inter = (pred_fg * target_fg).sum()
        union = pred_fg.sum() + target_fg.sum()
        dice = (2 * inter + 1e-6) / (union + 1e-6)
        
        return {"dice": dice}
    
    def training_step(self, batch, batch_idx):
        x, y, _ = batch
        
        # One-hot encode targets
        y_onehot = F.one_hot(y.squeeze(1).long(), num_classes=self.hparams.out_channels).permute(0, 4, 1, 2, 3).float()
        
        logits = self(x)
        loss = self.loss_fn(logits, y_onehot)
        metrics = self._compute_metrics(logits, y)
        
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_dice", metrics["dice"], prog_bar=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y, _ = batch
        
        y_onehot = F.one_hot(y.squeeze(1).long(), num_classes=self.hparams.out_channels).permute(0, 4, 1, 2, 3).float()
        
        logits = self(x)
        loss = self.loss_fn(logits, y_onehot)
        metrics = self._compute_metrics(logits, y)
        
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_dice", metrics["dice"], prog_bar=True)
        return loss
    
    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-6)
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "interval": "epoch"}}

# ===========================
# TRAIN
# ===========================
def train():
    print("\nTraining...")
    
    # Data
    dm = SimpleDataModule(
        Config.TRAIN_IMAGES_DIR,
        Config.TRAIN_LABELS_DIR,
        Config.INPUT_SIZE,
        Config.VAL_SPLIT,
        Config.BATCH_SIZE,
        Config.NUM_WORKERS
    )
    dm.setup()
    
    # Model
    model = SimpleDynUNet(
        in_channels=Config.IN_CHANNELS,
        out_channels=Config.OUT_CHANNELS,
        lr=Config.LR
    )
    
    print(f"\nModel: DynUNet")
    
    # Callbacks
    from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
    
    checkpoint = ModelCheckpoint(
        dirpath=Config.OUTPUT_DIR,
        filename="dynunet-{epoch:02d}-{val_dice:.4f}",
        monitor="val_dice",
        mode="max",
        save_top_k=2,
        verbose=True
    )
    
    early_stop = EarlyStopping(monitor="val_dice", patience=8, mode="max", verbose=True)
    
    # Trainer
    trainer = pl.Trainer(
        max_epochs=Config.MAX_EPOCHS,
        accelerator="auto",
        devices="auto",
        callbacks=[checkpoint, early_stop],
        precision="16-mixed",
        log_every_n_steps=10,
        accumulate_grad_batches=4
    )
    
    trainer.fit(model, dm)
    
    print("\n✓ Training done!")
    return trainer, model, dm

# ===========================
# INFERENCE
# ===========================
def inference(checkpoint_path, dm):
    print("\nInference...")
    
    model = SimpleDynUNet.load_from_checkpoint(checkpoint_path)
    model.eval()
    model.to(Config.DEVICE)
    
    test_files = sorted([f.name for f in Config.TEST_IMAGES_DIR.glob("*.tif")])
    test_dataset = SimpleDataset(Config.TEST_IMAGES_DIR, None, test_files)
    
    print(f"Processing {len(test_files)} volumes...")
    
    predictions = []
    
    for img, _, fid in tqdm(test_dataset):
        img_device = img.to(model.device)
        processed = dm.val_transform({"image": img_device, "label": torch.zeros_like(img_device)})
        inputs = processed["image"].unsqueeze(0)
        
        with torch.no_grad():
            logits = model(inputs)
            pred = torch.argmax(torch.softmax(logits, dim=1), dim=1)[0]
        
        # Resize to original
        orig_path = Config.TEST_IMAGES_DIR / f"{fid}.tif"
        with tifffile.TiffFile(str(orig_path)) as tif:
            orig_shape = tif.series[0].shape
        
        pred_resized = F.interpolate(
            pred.float().unsqueeze(0).unsqueeze(0),
            size=orig_shape,
            mode='nearest'
        ).squeeze().byte().cpu().numpy()
        
        save_path = Config.OUTPUT_DIR / f"{fid}.tif"
        tifffile.imwrite(str(save_path), pred_resized, compression='lzw')
        predictions.append(f"{fid}.tif")
    
    # Zip
    with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as z:
        for fname in tqdm(predictions):
            if os.path.exists(fname):
                z.write(fname)
                os.remove(fname)
    
    print("✓ Submission ready!")

# ===========================
# MAIN
# ===========================
if __name__ == "__main__":
    trainer, model, dm = train()
    
    # Find best checkpoint
    import re
    ckpts = list(Config.OUTPUT_DIR.glob("dynunet-*.ckpt"))
    if ckpts:
        pattern = re.compile(r"val_dice=?([0-9]+\.[0-9]+)")
        best = max(ckpts, key=lambda p: float(pattern.search(p.name).group(1)))
        print(f"\nBest: {best}")
        
        inference(str(best), dm)
    
    print("\n" + "="*70)
    print("✓ DONE!")
    print("="*70)
    print("\n📊 CURRENT SETUP:")
    print("   Model: DynUNet (simple, auto-architecture)")
    print("   Size: 160³")
    print("   Batch: 2 × 4 = 8 (with gradient accumulation)")
    print("   Augment: Just flips")
    print("   Expected: 0.54-0.57")
    print("\n💡 TO IMPROVE SCORE (Easy → Hard):")
    print("\n1. EASY WINS (+0.02-0.03):")
    print("   - Change INPUT_SIZE to (192, 192, 192)")
    print("   - Add: MT.RandRotated(...) in augmentations")
    print("   - Change LR to 2e-4")
    print("\n2. MEDIUM IMPROVEMENTS (+0.02-0.04):")
    print("   - Add TverskyLoss (alpha=0.3, beta=0.7) for better recall")
    print("   - Increase BATCH_SIZE to 4 if memory allows")
    print("   - Add dropout=0.2 in DynUNet")
    print("\n3. BIGGER CHANGES (+0.03-0.06):")
    print("   - Switch to SegResNet (proven better)")
    print("   - Train 2 models and ensemble predictions")
    print("   - Add test-time augmentation (TTA)")
    print("\n4. ADVANCED (+0.05-0.08):")
    print("   - Use SwinUNETR (best but complex)")
    print("   - Deep supervision = True")
    print("   - Post-processing (morphological operations)")
    print("="*70)

VESUVIUS - Simple DynUNet
Input: (160, 160, 160)
Batch: 2

Training...
Train: 644, Val: 162
Dataset: 644 volumes
Dataset: 162 volumes


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs



Model: DynUNet
Train: 644, Val: 162
Dataset: 644 volumes
Dataset: 162 volumes


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name    | Type       | Params | Mode 
-----------------------------------------------
0 | model   | DynUNet    | 16.5 M | train
1 | loss_fn | DiceCELoss | 0      | train
-----------------------------------------------
16.5 M    Trainable params
0         Non-trainable params
16.5 M    Total params
66.154    Total estimated model params size (MB)
144       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Monitored metric val_dice did not improve in the last 8 records. Best score: 0.761. Signaling Trainer to stop.
Epoch 12, global step 1053: 'val_dice' was not in top 2



✓ Training done!

Best: dynunet-epoch=04-val_dice=0.7609.ckpt

Inference...
Dataset: 1 volumes
Processing 1 volumes...


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

✓ Submission ready!

✓ DONE!

📊 CURRENT SETUP:
   Model: DynUNet (simple, auto-architecture)
   Size: 160³
   Batch: 2 × 4 = 8 (with gradient accumulation)
   Augment: Just flips
   Expected: 0.54-0.57

💡 TO IMPROVE SCORE (Easy → Hard):

1. EASY WINS (+0.02-0.03):
   - Change INPUT_SIZE to (192, 192, 192)
   - Add: MT.RandRotated(...) in augmentations
   - Change LR to 2e-4

2. MEDIUM IMPROVEMENTS (+0.02-0.04):
   - Add TverskyLoss (alpha=0.3, beta=0.7) for better recall
   - Increase BATCH_SIZE to 4 if memory allows
   - Add dropout=0.2 in DynUNet

3. BIGGER CHANGES (+0.03-0.06):
   - Switch to SegResNet (proven better)
   - Train 2 models and ensemble predictions
   - Add test-time augmentation (TTA)

4. ADVANCED (+0.05-0.08):
   - Use SwinUNETR (best but complex)
   - Deep supervision = True
   - Post-processing (morphological operations)
